# 02 - Baseline Encoding Model

Loads video stimuli, extracts AlexNet activations, and trains a ridge
regression encoding model to predict fMRI responses per ROI.

**Fix applied:** the Algonauts video folder contains 1102 files (1000
train + 102 held-out test videos with no fMRI labels). This notebook
now explicitly keeps only the first 1000, and validates the shape
before running any regression so a mismatch never fails silently again.

In [ ]:
!git clone https://github.com/sossyh/ffa-dnn-ablation.git
%cd ffa-dnn-ablation

In [ ]:
!pip install nilearn decord python-dotenv --quiet

## Load dataset link from Colab Secrets

Requires a secret named `DROPBOX_DATASET_LINK` set via the key icon in
the left sidebar, with notebook access turned on.

In [ ]:
import os
from google.colab import userdata

os.environ["DROPBOX_DATASET_LINK"] = userdata.get("DROPBOX_DATASET_LINK")
print("Link loaded:", os.environ["DROPBOX_DATASET_LINK"] is not None)

In [ ]:
import sys
sys.path.append(".")

from src.data_loading import (
    download_algonauts_data,
    load_all_rois,
    get_video_list,
    load_video_frames,
)
from src.models.alexnet_wrapper import AlexNetWrapper
from src.encoding_model import train_and_evaluate_encoding_model, region_mean_accuracy

In [ ]:
download_algonauts_data(data_dir="data/raw")

## Load fMRI ROI data

In [ ]:
fmri_dir = "data/raw/participants_data_v2021/mini_track"
subject = "sub01"

roi_data = load_all_rois(fmri_dir, subject)

for roi_name, data in roi_data.items():
    print(f"{roi_name}: {data.shape}")

NUM_TRAIN_VIDEOS = roi_data["FFA"].shape[0]
print(f"\nNumber of training videos (from fMRI data): {NUM_TRAIN_VIDEOS}")

## Load video list

The video folder contains 1102 files: 1000 training videos with fMRI
labels, plus 102 held-out test videos from the Algonauts challenge that
have no fMRI data here. We keep only the first `NUM_TRAIN_VIDEOS`
(matched to the fMRI data itself, not hardcoded), so this stays correct
even if you switch ROI files or subjects later.

In [ ]:
video_dir = "data/raw/AlgonautsVideos268_All_30fpsmax"
video_paths = get_video_list(video_dir)

print(f"Total video files found: {len(video_paths)}")

video_paths = video_paths[:NUM_TRAIN_VIDEOS]

assert len(video_paths) == NUM_TRAIN_VIDEOS, (
    f"Video count ({len(video_paths)}) does not match fMRI row count "
    f"({NUM_TRAIN_VIDEOS}). Check the video folder contents before continuing."
)

print(f"Using {len(video_paths)} training videos (matched to fMRI data)")
print("First 3 paths:", video_paths[:3])
print("Last 3 paths:", video_paths[-3:])

## Extract AlexNet activations for every video

Cached to disk so this expensive step never needs to be recomputed
unless the video count changes (in which case the cache is
automatically invalidated and rebuilt, so the mismatch from before
can't happen silently again).

In [ ]:
import numpy as np
from tqdm import tqdm

os.makedirs("data/processed", exist_ok=True)
activation_cache_path = "data/processed/alexnet_activations.npz"

need_to_extract = True

if os.path.exists(activation_cache_path):
    print("Found cached activations, checking they match current video count...")
    cached = np.load(activation_cache_path, allow_pickle=True)
    all_activations = cached["all_activations"].item()
    cached_num_videos = next(iter(all_activations.values())).shape[0]

    if cached_num_videos == len(video_paths):
        print(f"Cache matches ({cached_num_videos} videos). Using cached activations.")
        need_to_extract = False
    else:
        print(
            f"Cache has {cached_num_videos} videos but current list has "
            f"{len(video_paths)}. Cache is stale, re-extracting."
        )

if need_to_extract:
    print("Extracting activations (this will take a while)...")
    alexnet = AlexNetWrapper()

    all_activations = {layer: [] for layer in alexnet.layers}

    for video_path in tqdm(video_paths):
        frames = load_video_frames(video_path, num_frames=8)
        layer_acts = alexnet.extract(frames)
        for layer in alexnet.layers:
            all_activations[layer].append(layer_acts[layer])

    for layer in all_activations:
        all_activations[layer] = np.stack(all_activations[layer])

    np.savez(activation_cache_path, all_activations=all_activations)
    print("Saved activations to", activation_cache_path)

for layer, acts in all_activations.items():
    print(f"{layer}: {acts.shape}")

## Sanity check before running any regression

This assert is what would have caught the original bug immediately,
instead of failing three cells later with a cryptic IndexError.

In [ ]:
for layer, acts in all_activations.items():
    assert acts.shape[0] == NUM_TRAIN_VIDEOS, (
        f"Layer {layer} has {acts.shape[0]} rows but fMRI data has "
        f"{NUM_TRAIN_VIDEOS} rows. These must match before running regression."
    )

for roi_name, data in roi_data.items():
    assert data.shape[0] == NUM_TRAIN_VIDEOS, (
        f"ROI {roi_name} has {data.shape[0]} rows, expected {NUM_TRAIN_VIDEOS}."
    )

print("All shapes match. Safe to proceed.")

## Train the encoding model: one layer, one region, as a first test

In [ ]:
X = all_activations["conv5"]
Y = roi_data["FFA"]

voxel_scores = train_and_evaluate_encoding_model(X, Y)
print("Mean FFA accuracy (conv5):", region_mean_accuracy(voxel_scores))

## Full sweep: every layer x every region

In [ ]:
import pandas as pd

os.makedirs("results/tables/alexnet", exist_ok=True)

results = []

for layer_name, X in all_activations.items():
    for region_name, Y in roi_data.items():
        voxel_scores = train_and_evaluate_encoding_model(X, Y)
        mean_acc = region_mean_accuracy(voxel_scores)
        results.append({"layer": layer_name, "region": region_name, "accuracy": mean_acc})
        print(f"{layer_name} -> {region_name}: {mean_acc:.4f}")

results_df = pd.DataFrame(results)
results_df.to_csv("results/tables/alexnet/baseline_accuracy.csv", index=False)
results_df.head()

This CSV (`results/tables/alexnet/baseline_accuracy.csv`) is your saved
baseline. Everything in the ablation notebook later gets compared
against these numbers, so keep this file around (commit it to the repo
once you're happy with it).